In [ ]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
while not (root / "data" / "raw").exists() and root != root.parent:
    root = root.parent
raw_dir = root / "data" / "raw"

frames = []
for path in sorted(raw_dir.glob("[0-9][0-9][0-9][0-9].csv")):
    d = pd.read_csv(path)
    d["file_year"] = int(path.stem)
    frames.append(d)

raw_all = pd.concat(frames, ignore_index=True)
print(len(frames), "files loaded", raw_all.shape)

In [ ]:
#filtering the rows 
clean = raw_all.copy()
print("start:", len(clean))

clean = clean[~clean["tourney_level"].isin(["D", "O"])]
print("after Davis Cup and Olympics:", len(clean))

keep_a = ["Doha", "Los Cabos"]
clean = clean[(clean["tourney_level"] != "A") | clean["tourney_name"].isin(keep_a)]
print("after team events and Next Gen:", len(clean))

clean = clean[~clean["score"].str.contains("W/O|DEF", na=False)]
print("after walkovers and defaults:", len(clean))

In [ ]:
#cleaning 
clean = clean.copy()
clean["indoor"] = clean["indoor"].replace({"0": "O", 0: "O"})
clean["date"] = pd.to_datetime(clean["tourney_date"].astype(str), format="%Y%m%d")
clean["retired"] = clean["score"].str.contains("RET", na=False)
clean["event_no"] = clean["tourney_id"].astype(str).str.split("-").str[1]

In [ ]:
#check cleaning
print(clean["indoor"].value_counts(dropna=False).to_string())
print("missing surface:", clean["surface"].isna().sum())
print("missing event_no:", clean["event_no"].isna().sum())
print("missing best_of:", clean["best_of"].isna().sum())
print(clean["date"].min(), clean["date"].max())

In [ ]:
# mark 14 tournaments we will use 
t = pd.read_csv(root / "data" / "own" / "tournaments.csv", dtype={"event_no": str})
clean["analysed"] = clean["event_no"].isin(t["event_no"]) & clean["tourney_level"].isin(["G", "M", "F"])

print(clean["analysed"].sum())
print(clean[clean["analysed"]].groupby("file_year").size().to_string())

In [ ]:
other = [c for c in clean.columns if c != "match_num"]
same = clean[clean.duplicated(subset=other, keep=False)]
print(len(same))
print(same.groupby(["tourney_level", "tourney_name"]).size().to_string())

In [ ]:
print("empty match_num:", raw_all["match_num"].isna().sum())
print(raw_all["indoor"].value_counts(dropna=False).to_string())
print(raw_all[raw_all["tourney_level"] == "A"]["tourney_name"].value_counts().to_string())

In [ ]:
key = ["tourney_id", "round", "winner_id", "loser_id"]
rep = clean[clean.duplicated(subset=key + ["file_year"], keep=False)]
print("repeated rows:", len(rep))
print(rep.groupby(["tourney_level", "tourney_name"]).size().to_string())

other = [c for c in clean.columns if c != "match_num"]
same = clean[clean.duplicated(subset=other, keep=False)]
print("identical apart from match_num:", len(same))